# Basic Financial Retrieval System

In [1]:
from pprint import pprint
import os
import getpass

In [2]:
# Load the PDF document
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("..\\data\\raw\\_10-K-2025-As-Filed.pdf")
reader = loader.load()


print(len(reader))

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_18756\998447580.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


80


In [3]:
# split the document into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(reader)

pprint([doc.page_content[:100] for doc in docs[:3]])

['UNITED STATES\n'
 'SECURITIES AND EXCHANGE COMMISSION\n'
 'Washington, D.C. 20549\n'
 'FORM 10-K\n'
 '(Mark One)\n'
 '☒    AN',
 'Title of each class\n'
 'Trading \n'
 'symbol(s) Name of each exchange on which registered\n'
 'Common Stock, $0.00',
 'Indicate by check mark whether the Registrant (1)\xa0has filed all reports '
 'required to be filed by Sect']


In [4]:
# API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")

In [ ]:
# Embed the chunks and store them in a vector database
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_chroma import Chroma

embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')

vector_store = Chroma.from_documents(
    documents=docs,
    collection_name="apple-10k",
    embedding=embeddings
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# Retrieving from Vector DB
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})

query = "What are the company's main risk factors?"
results = retriever._get_relevant_documents(query, run_manager=True)

print(results[0].page_content)

Item 1A. Risk Factors
The following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of 
operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these 
risks. Statements in this section are based on the Company’s beliefs and opinions regarding matters that could materially 
adversely affect the Company in the future and are not representations as to whether such matters have or have not occurred 
previously. The risks and uncertainties described below are not exhaustive and should not be considered a complete statement 
of all potential risks or uncertainties that the Company faces or may face in the future.
This section should be read in conjunction with Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition 
and Results of Operations” and the consolidated financial statements and accompanying notes in Part II, Item 8, “Financial
